# Lab 1: Functional API Basics

## Topic 3: Flexible ML Architectures with Vibe Coding

In this lab, we move beyond the Sequential API and learn the **Keras Functional API**, which allows us to build complex model architectures with branching, merging, and multiple inputs/outputs.

### What You Will Learn
- Limitations of the Sequential API
- Building models with `keras.Input` and `keras.Model`
- Visualizing architectures with `keras.utils.plot_model()`
- Creating branching architectures with parallel Conv2D paths

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt

from keras import layers, Model, Input
from keras.utils import plot_model

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.backend.backend()}")

## 1. Load and Prepare CIFAR-10 Dataset

In [ ]:
# Load CIFAR-10
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize pixel values to [0, 1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Class names for CIFAR-10
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Number of classes: {len(CLASS_NAMES)}")

In [ ]:
# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(CLASS_NAMES[y_train[i][0]])
    ax.axis("off")
plt.suptitle("CIFAR-10 Sample Images", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Sequential API Review

The Sequential API is simple and intuitive: you stack layers one after another. However, it has limitations:
- Only supports a single input and single output
- Cannot have branching or merging paths
- Cannot share layers between different parts of a model

In [ ]:
# Build a CNN with the Sequential API
sequential_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
], name="sequential_cnn")

sequential_model.summary()

In [ ]:
# Compile and train the Sequential model
sequential_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

sequential_history = sequential_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Evaluate the Sequential model
seq_loss, seq_acc = sequential_model.evaluate(X_test, y_test, verbose=0)
print(f"Sequential Model - Test Loss: {seq_loss:.4f}, Test Accuracy: {seq_acc:.4f}")

## 3. Functional API: Rebuilding the Same CNN

The Functional API uses two key building blocks:
- `keras.Input(shape=...)` - defines the input tensor
- `keras.Model(inputs=..., outputs=...)` - defines the model from input to output

Each layer is called as a function on its input tensor, and returns an output tensor.

In [ ]:
# Build the same CNN with the Functional API
inputs = Input(shape=(32, 32, 3), name="image_input")

x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
x = layers.Flatten()(x)
x = layers.Dense(64, activation="relu")(x)
outputs = layers.Dense(10, activation="softmax", name="classification")(x)

functional_model = Model(inputs=inputs, outputs=outputs, name="functional_cnn")
functional_model.summary()

In [ ]:
# Visualize the model architecture
plot_model(
    functional_model,
    show_shapes=True,
    show_layer_names=True,
    to_file="functional_cnn.png",
    dpi=100
)

In [ ]:
# Compile and train the Functional model
functional_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

functional_history = functional_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Evaluate the Functional model
func_loss, func_acc = functional_model.evaluate(X_test, y_test, verbose=0)
print(f"Functional Model - Test Loss: {func_loss:.4f}, Test Accuracy: {func_acc:.4f}")
print(f"Sequential Model - Test Loss: {seq_loss:.4f}, Test Accuracy: {seq_acc:.4f}")
print("\nBoth models have the same architecture, so performance should be comparable.")

## 4. Branching Architecture: Parallel Conv2D Paths

Now let's demonstrate something the Sequential API **cannot** do: a model with two parallel convolutional paths that are merged together.

This is sometimes called a "multi-branch" or "inception-style" architecture:
- **Branch A**: Uses 3x3 convolutions (captures local features)
- **Branch B**: Uses 5x5 convolutions (captures wider features)
- Both branches are merged via `Concatenate`

In [ ]:
# Build a branching model with two parallel Conv2D paths
inputs = Input(shape=(32, 32, 3), name="image_input")

# Branch A: 3x3 convolutions (local features)
branch_a = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="branch_a_conv1")(inputs)
branch_a = layers.MaxPooling2D((2, 2), name="branch_a_pool1")(branch_a)
branch_a = layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="branch_a_conv2")(branch_a)
branch_a = layers.MaxPooling2D((2, 2), name="branch_a_pool2")(branch_a)

# Branch B: 5x5 convolutions (wider features)
branch_b = layers.Conv2D(32, (5, 5), activation="relu", padding="same", name="branch_b_conv1")(inputs)
branch_b = layers.MaxPooling2D((2, 2), name="branch_b_pool1")(branch_b)
branch_b = layers.Conv2D(32, (5, 5), activation="relu", padding="same", name="branch_b_conv2")(branch_b)
branch_b = layers.MaxPooling2D((2, 2), name="branch_b_pool2")(branch_b)

# Merge branches via Concatenate
merged = layers.Concatenate(name="merge_branches")([branch_a, branch_b])

# Classification head
x = layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="merged_conv")(merged)
x = layers.Flatten(name="flatten")(x)
x = layers.Dense(64, activation="relu", name="dense_1")(x)
x = layers.Dropout(0.3, name="dropout")(x)
outputs = layers.Dense(10, activation="softmax", name="classification")(x)

branching_model = Model(inputs=inputs, outputs=outputs, name="branching_cnn")
branching_model.summary()

In [ ]:
# Visualize the branching architecture
plot_model(
    branching_model,
    show_shapes=True,
    show_layer_names=True,
    to_file="branching_cnn.png",
    dpi=100
)

In [ ]:
# Compile and train the branching model
branching_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

branching_history = branching_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Evaluate the branching model
branch_loss, branch_acc = branching_model.evaluate(X_test, y_test, verbose=0)
print(f"Branching Model - Test Loss: {branch_loss:.4f}, Test Accuracy: {branch_acc:.4f}")

## 5. Compare All Three Models

In [ ]:
# Compare training histories
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].plot(sequential_history.history["val_accuracy"], label="Sequential", linestyle="--")
axes[0].plot(functional_history.history["val_accuracy"], label="Functional")
axes[0].plot(branching_history.history["val_accuracy"], label="Branching")
axes[0].set_title("Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss comparison
axes[1].plot(sequential_history.history["val_loss"], label="Sequential", linestyle="--")
axes[1].plot(functional_history.history["val_loss"], label="Functional")
axes[1].plot(branching_history.history["val_loss"], label="Branching")
axes[1].set_title("Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Model Comparison: Sequential vs Functional vs Branching", fontsize=14)
plt.tight_layout()
plt.show()

# Summary table
print("\n" + "="*55)
print(f"{'Model':<20} {'Test Loss':<15} {'Test Accuracy':<15}")
print("="*55)
print(f"{'Sequential':<20} {seq_loss:<15.4f} {seq_acc:<15.4f}")
print(f"{'Functional':<20} {func_loss:<15.4f} {func_acc:<15.4f}")
print(f"{'Branching':<20} {branch_loss:<15.4f} {branch_acc:<15.4f}")
print("="*55)